In [1]:
"""
네트워크분석 기반 XGBoost 피처 생성

이 노트북은 두 부분으로 구성된다.

PART A. 정적(static) 매개중심성 피처
  network/network_betweenness.csv(링크 단위, 25,401행)를
  output/segment_link_mapping.csv(link_id -> segment_id, direction, length_m)로
  LINK_ID 기준 join 후, 우리 segment_key(=segment_id+direction) 기준으로
  길이가중평균 재집계해 betweenness_pre / betweenness_during을 만든다.
  network 팀 자체 segment_id 롤업은 우리 정의(방향 구분, 인접 정거장 순서)와
  달라서 쓰지 않고, LINK_ID 레벨에서 우리가 직접 재집계한다
  (706/706 링크 완전 매칭 확인됨).

  centrality_delta(=betweenness_during - betweenness_pre)는 만들지 않는다.
  bc_free_flow(제한속도 이상 시나리오)와 bc_its_realtime(단일 스냅샷 실측)의
  차이가 "공사 파급력"이 아니라 "이상적 속도 대비 실제 혼잡도"에 가깝다는 게
  확인되어(간선도로일수록 이 격차가 원래 크게 남), 이미 갖고 있는 훨씬 안정적인
  실측 혼잡도 피처(interval_summary_stats, is_bottleneck_slot, lane_remain_ratio)
  와 중복이라 제외했다. betweenness_during은 일단 남겨두고 추후 모델 성능을
  보고 유지 여부를 재판단한다.

PART B. 시간가변(time-varying) 공사 통제 비율 피처 (lane_remain_ratio)
  network_betweenness.csv의 lane_remain_ratio는 단일 스냅샷(2026-07-08 기준)이라,
  우리 학습기간(2024-10 ~ 2026-07)에 그대로 붙이면 착공 전(자유흐름) 구간에도
  "차로 50% 폐쇄" 같은 모순된 값이 붙는 문제가 있어(지난 대화에서 확인),
  대신 다음 원본을 직접 파싱해 (segment_id, date) 단위 시간가변 비율을 만든다.
    - data/트램_공구별_통제현황.xlsx (주 소스, 1~7/9/10/12~14공구 커버)
    - data/공사개요(총괄) 표 (보조 소스, 통제현황에 없는 8/11공구 fallback 용)
    - data/공구_구간_착공일.csv (공구 <-> segment_id 다리)
    - data/daejeon_link.csv (공구+도로명별 실제 차로 수 룩업 - 폐쇄 차로 수만
      명시되고 전체 차로 수가 없는 텍스트를 복구하는 데 사용)

출력:
  output/features/network_features.parquet
    segment_key, betweenness_pre, betweenness_during, road_rank, lanes
  output/features/construction_lane_ratio_daily.parquet
    segment_id, date, lane_remain_ratio
"""

import re
from pathlib import Path

import pandas as pd
import polars as pl

NETWORK_BC_PATH = "./network/network_betweenness.csv"
SEGMENT_LINK_MAPPING_PATH = "./output/segment_link_mapping.csv"
GONGU_START_PATH = "./data/공구_구간_착공일.csv"
CONTROL_STATUS_XLSX_PATH = "./data/트램_공구별_통제현황.xlsx"
SPEED_PATH = "./output/segment_weighted_speed_final.parquet"

OUTPUT_DIR = Path("./output/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# ==================================================================
# PART A-1. 링크 단위 BC + 우리 구간 매핑 join
# ==================================================================

net = pl.read_csv(NETWORK_BC_PATH)
seg_map = pl.read_csv(SEGMENT_LINK_MAPPING_PATH)

print(f"network_betweenness.csv: {net.shape}")
print(f"segment_link_mapping.csv: {seg_map.shape}")

target_link_ids = seg_map["link_id"].unique()
matched = net.filter(pl.col("LINK_ID").is_in(target_link_ids))
print(f"우리 구간이 쓰는 link_id 중 network 파일에 있는 건수: {matched.height} / {target_link_ids.len()}")

joined = seg_map.select(["link_id", "segment_id", "direction", "length_m"]).join(
    net.select(["LINK_ID", "bc_free_flow", "bc_its_realtime", "bc_its_vs_free", "ROAD_RANK", "LANES"]),
    left_on="link_id",
    right_on="LINK_ID",
    how="inner",
).with_columns((pl.col("segment_id") + "_" + pl.col("direction")).alias("segment_key"))

print(f"join 결과: {joined.shape}")
joined.head()

network_betweenness.csv: (25401, 23)
segment_link_mapping.csv: (729, 18)
우리 구간이 쓰는 link_id 중 network 파일에 있는 건수: 706 / 706
join 결과: (729, 10)


C:\Users\6152\AppData\Local\Temp\ipykernel_26772\2769386682.py:12: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  matched = net.filter(pl.col("LINK_ID").is_in(target_link_ids))


link_id,segment_id,direction,length_m,bc_free_flow,bc_its_realtime,bc_its_vs_free,ROAD_RANK,LANES,segment_key
i64,str,str,f64,f64,f64,f64,i64,i64,str
1830024203,"""SEG_07_207_208""","""AB""",12.035,0.000569,0.001175,0.000606,104,2,"""SEG_07_207_208_AB"""
1860094400,"""SEG_21_221_222""","""AB""",58.852,0.000423,0.002547,0.002124,104,2,"""SEG_21_221_222_AB"""
1840004402,"""SEG_04_204_205""","""BA""",154.011,0.027699,0.005171,-0.022528,103,3,"""SEG_04_204_205_BA"""
1850130000,"""SEG_17_217_218""","""BA""",328.343,0.02566,0.005872,-0.019788,104,3,"""SEG_17_217_218_BA"""
1850003800,"""SEG_33_233_234""","""BA""",215.599,0.023699,0.011684,-0.012016,103,4,"""SEG_33_233_234_BA"""


In [3]:
# ==================================================================
# PART A-2. segment_key 단위 재집계
# ==================================================================
# - betweenness_pre / betweenness_during : 길이가중평균 (V_segment와 동일한 방식,
#   파이프라인 전체 일관성 유지. BC가 특정 짧은 링크에서 튀는 문제를
#   단순평균보다 완화하면서도 극단값 하나에 좌우되는 것을 막아줌)
# - road_rank : 구간 내 최빈값(mode)
# - lanes : 구간 내 최댓값(max) - 그 구간이 가질 수 있는 최대 용량 기준
#
# centrality_delta(=betweenness_during - betweenness_pre)는 제외한다.
# bc_free_flow(제한속도 이상 시나리오)와 bc_its_realtime(2026-07-07 단일 스냅샷
# 실측)의 차이는 "공사로 인한 파급력 변화"가 아니라 "이상적 속도 대비 실제
# 혼잡도"에 가깝고(간선도로일수록 원래 이 격차가 크게 나옴), 우리가 이미
# 638일치 실측으로 훨씬 안정적으로 갖고 있는 혼잡도 정보(interval_summary_stats,
# is_bottleneck_slot, lane_remain_ratio)와 중복된다는 게 확인되어 제외.
# betweenness_pre(구조적 중요도, 우리 속도 데이터에 없는 정보)는 유지하고,
# betweenness_during은 일단 남겨두되 추후 모델 성능을 보고 유지 여부를 재판단한다.


def _weighted_mean(value_col: str, weight_col: str = "length_m") -> pl.Expr:
    return (pl.col(value_col) * pl.col(weight_col)).sum() / pl.col(weight_col).sum()


network_features = (
    joined.group_by("segment_key")
    .agg(
        [
            _weighted_mean("bc_free_flow").alias("betweenness_pre"),
            _weighted_mean("bc_its_realtime").alias("betweenness_during"),
            pl.col("ROAD_RANK").mode().first().alias("road_rank"),
            pl.col("LANES").max().alias("lanes"),
        ]
    )
    .select(
        [
            "segment_key",
            "betweenness_pre",
            "betweenness_during",
            "road_rank",
            "lanes",
        ]
    )
    .sort("segment_key")
)

network_features.write_parquet(OUTPUT_DIR / "network_features.parquet")

print(f"segment_key 수: {network_features.height}")
print(network_features.sort("betweenness_during", descending=True).head(10))
print(f"\n저장 완료: {(OUTPUT_DIR / 'network_features.parquet').resolve()}")


segment_key 수: 90
shape: (10, 5)
┌───────────────────┬─────────────────┬────────────────────┬───────────┬───────┐
│ segment_key       ┆ betweenness_pre ┆ betweenness_during ┆ road_rank ┆ lanes │
│ ---               ┆ ---             ┆ ---                ┆ ---       ┆ ---   │
│ str               ┆ f64             ┆ f64                ┆ i64       ┆ i64   │
╞═══════════════════╪═════════════════╪════════════════════╪═══════════╪═══════╡
│ SEG_23_223_224_AB ┆ 0.026311        ┆ 0.020624           ┆ 104       ┆ 5     │
│ SEG_13_213_214_AB ┆ 0.015605        ┆ 0.017426           ┆ 104       ┆ 5     │
│ SEG_35_235_236_BA ┆ 0.037528        ┆ 0.016684           ┆ 103       ┆ 5     │
│ SEG_14_214_215_BA ┆ 0.016817        ┆ 0.016665           ┆ 104       ┆ 5     │
│ SEG_13_213_214_BA ┆ 0.014368        ┆ 0.016114           ┆ 104       ┆ 5     │
│ SEG_35_235_236_AB ┆ 0.036282        ┆ 0.016041           ┆ 103       ┆ 5     │
│ SEG_36_236_237_AB ┆ 0.036106        ┆ 0.014816           ┆ 103       ┆ 4  

In [4]:
# ==================================================================
# PART B-1. 공구 <-> segment_id 다리 로드
# ==================================================================
# 공구_구간_착공일.csv는 14개 공구 전체에 대해 어느 segment_id가 속하는지를 이미
# 가지고 있으므로, 트램_공구별_통제현황.xlsx의 "공구" 번호만으로 다리를 놓는다.
# (도로명 복합표기/괄호 등을 파싱해야 했던 network 팀의 도로명 매칭 방식을 피함)

gongu_map = pd.read_csv(GONGU_START_PATH, encoding="utf-8-sig")
gongu_map["착공일"] = pd.to_datetime(gongu_map["착공일"])

print(gongu_map.shape)
print(gongu_map.head())
print("\n공구별 segment_id 개수:")
print(gongu_map.groupby("공구")["segment_id"].count())

(45, 3)
   공구      segment_id        착공일
0   1  SEG_43_242_243 2024-12-06
1   1  SEG_44_243_244 2024-12-06
2   2  SEG_41_212_241 2024-12-26
3   2  SEG_42_241_242 2024-12-26
4   3  SEG_12_212_213 2025-09-15

공구별 segment_id 개수:
공구
1     2
2     2
3     3
4     3
5     3
6     2
7     4
8     5
9     4
10    2
11    1
12    6
13    5
14    3
Name: segment_id, dtype: int64


In [5]:
# ==================================================================
# PART B-2. 트램_공구별_통제현황.xlsx 로드/정리
# ==================================================================
# 시트 "공구별 통제현황"만 사용 (버스전용차로 유예 시트는 이번 피처와 무관해서 제외).
# "상태(활성/종료)" 컬럼은 사용하지 않음 - 통제시작일~통제종료일 자체가 이미
# 조사 시점(2026-07-08) 기준 활성/종료 여부를 담고 있으므로, 임의 시점 t와 날짜 범위를
# 직접 비교하면 더 정확하게 판정할 수 있음.

raw = pd.read_excel(CONTROL_STATUS_XLSX_PATH, sheet_name=0, skiprows=3)
raw.columns = [
    "gongu", "road_name", "sub_section", "location_text",
    "start_date", "end_date", "time_window", "lane_method_text", "status", "note",
]
raw = raw.dropna(subset=["gongu"]).copy()

# "gongu" 컬럼엔 "1공구" 같은 실제 데이터 행 외에, 파일 하단의 범례/주석 텍스트도
# 섞여 있어 숫자 추출이 실패(NaN)한다 -> 추출 실패 행은 데이터가 아니므로 제거.
raw["gongu"] = raw["gongu"].astype(str).str.extract(r"(\d+)")[0]
raw = raw.dropna(subset=["gongu"]).copy()
raw["gongu"] = raw["gongu"].astype(int)
raw = raw[raw["gongu"].between(1, 14)].copy()

raw["start_date"] = pd.to_datetime(raw["start_date"], errors="coerce")
raw["end_date"] = pd.to_datetime(raw["end_date"], errors="coerce")

print(f"통제현황 행 수: {len(raw)}")
print(f"날짜 결측 행 수: {raw['start_date'].isna().sum() + raw['end_date'].isna().sum()}")
print(f"등장하는 공구: {sorted(raw['gongu'].unique())}")
print(f"통제현황에 없는 공구(8/11공구 누락 예상): {sorted(set(range(1,15)) - set(raw['gongu'].unique()))}")
raw[["gongu", "road_name", "sub_section", "start_date", "end_date", "lane_method_text"]]


통제현황 행 수: 25
날짜 결측 행 수: 0
등장하는 공구: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(14)]
통제현황에 없는 공구(8/11공구 누락 예상): [8, 11]


,gongu,road_name,sub_section,start_date,end_date,lane_method_text
0,1,신탄진로,NaN,2025-03-25,2025-06-30,편도3차로 중 1차로
1,1,계족로,1구간,2025-05-08,2026-11-30,부분통제
2,1,계족로,2구간,2025-08-25,2026-11-30,부분통제
3,1,신탄진로,3구간,2026-01-15,2026-11-30,부분통제
4,2,계족로,1~4구간,2026-05-08,2026-08-31,편도3~5차로 중 1차로
5,2,계족로,법동네거리 일원,2025-06-24,2026-06-30,법동방향 2개차로
6,2,한밭대로,NaN,2026-02-20,2026-03-14,왕복10차로중2 / 왕복8차로중2 / 편도5차로중2
7,3,한밭대로,NaN,2026-06-01,2026-12-31,왕복 11개차로 중 편도 3개차로
8,4,한밭대로·대덕대로,NaN,2026-03-16,2026-12-31,편도 3~4개차로 중 1개차로
9,5,대덕대로,NaN,2026-06-08,2027-10-31,왕복 9개차로 중 2개차로


In [6]:
# ==================================================================
# PART B-2b. road_name(공구 기준) -> 실제 차로 수(최빈값) 룩업 테이블
# ==================================================================
# 통제방법(차로) 텍스트 중 "법동방향 2개차로"처럼 폐쇄 차로 수만 있고
# 전체 차로 수가 없는 경우, data/daejeon_link.csv(LINK_ID별 LANES)에서
# 그 공구가 담당하는 segment_id들의 링크 중 도로명이 일치하는 링크들의
# 실제 차로 수(최빈값)를 가져와 전체 차로 수로 사용한다.

DAEJEON_LINK_PATH = "./data/daejeon_link.csv"

seg_map_pd = pd.read_csv(SEGMENT_LINK_MAPPING_PATH, encoding="utf-8")
link_lanes = pd.read_csv(DAEJEON_LINK_PATH, encoding="utf-8")[["LINK_ID", "LANES"]]

seg_map_with_lanes = seg_map_pd.merge(link_lanes, left_on="link_id", right_on="LINK_ID", how="left")
seg_map_with_lanes = seg_map_with_lanes.merge(gongu_map[["공구", "segment_id"]], on="segment_id", how="left")


def clean_road_name(name: str) -> str:
    """복합 도로명(·) 분리 및 괄호 내용 제거 - network 팀 v2 파서와 동일한 정리 로직"""
    if not isinstance(name, str):
        return name
    name = re.sub(r"\([^)]*\)", "", name)
    return name.split("·")[0].strip()


def lookup_total_lanes(gongu: int, road_name: str):
    road_clean = clean_road_name(road_name)
    sub = seg_map_with_lanes[
        (seg_map_with_lanes["공구"] == gongu) & (seg_map_with_lanes["road_name"] == road_clean)
    ]
    if sub.empty or sub["LANES"].dropna().empty:
        return None
    return int(sub["LANES"].mode().iloc[0])


print("차로 수 룩업 테이블 준비 완료")


차로 수 룩업 테이블 준비 완료


In [7]:
# ==================================================================
# PART B-3. 차로 잔여비율 파서 (3단계)
# ==================================================================
# Stage 1: "왕복/편도 N차로 중 M차로" 완전 패턴에서 총차로(N)/폐쇄차로(M) 추출.
#   범위("3~4차로")는 보수적으로 총차로=작은값, 폐쇄차로=큰값 채택.
#   "A / B / C"처럼 여러 대안이 있으면 각각 파싱해 최솟값(가장 심각한 값) 채택.
#   2번째 숫자 뒤에 단위(차로/차선)가 생략된 경우("~중2")도 잡히도록 완화.
# Stage 2: Stage 1이 실패했지만 "N개차로"처럼 폐쇄 차로 수만 명시된 경우,
#   전체 차로 수를 daejeon_link.csv 기반 룩업 테이블(공구+도로명)에서
#   가져와 비율을 계산.
# Stage 3(fallback): 그래도 실패하면(숫자 정보 자체가 텍스트에 없음, 예: "부분통제")
#   보수적 기본값(0.5, network 팀과 동일한 관습) 적용.
# 주의: 자동 파싱 결과는 100% 정확하지 않을 수 있으므로 아래 QA 출력으로 육안 검증 필요.
# 특히 stage2는 "1~4차로 단계별통제"처럼 문맥상 모호한 숫자를 폐쇄차로로 오인할 수 있음.

LANE_PATTERN_FULL = re.compile(r"(\d+)(?:~(\d+))?개?차[로선]중.*?(\d+)(?:~(\d+))?개?(?:차[로선])?")
CLOSED_ONLY_PATTERN = re.compile(r"(\d+)개?차[로선]")
FALLBACK_RATIO = 0.5


def parse_single_full(text: str):
    if not isinstance(text, str):
        return None
    t = text.replace(" ", "")
    m = LANE_PATTERN_FULL.search(t)
    if not m:
        return None
    total_lo, total_hi, closed_lo, closed_hi = m.groups()
    total = int(total_lo)
    closed = int(closed_hi) if closed_hi else int(closed_lo)
    if total <= 0 or closed > total:
        return None
    return round((total - closed) / total, 4)


def parse_stage1(text: str):
    if not isinstance(text, str):
        return None
    candidates = [parse_single_full(part) for part in text.split("/")]
    candidates = [c for c in candidates if c is not None]
    return min(candidates) if candidates else None


def parse_stage2(text: str, gongu: int, road_name: str):
    if not isinstance(text, str):
        return None
    t = text.replace(" ", "")
    matches = CLOSED_ONLY_PATTERN.findall(t)
    if not matches:
        return None
    closed = int(matches[-1])
    total = lookup_total_lanes(gongu, road_name)
    if total is None or total <= 0 or closed > total:
        return None
    return round((total - closed) / total, 4)


def parse_lane_ratio(row):
    r1 = parse_stage1(row["lane_method_text"])
    if r1 is not None:
        return pd.Series([r1, "stage1_full_text"])
    r2 = parse_stage2(row["lane_method_text"], row["gongu"], row["road_name"])
    if r2 is not None:
        return pd.Series([r2, "stage2_lane_lookup"])
    return pd.Series([FALLBACK_RATIO, "fallback_default"])


raw[["lane_remain_ratio", "parse_source"]] = raw.apply(parse_lane_ratio, axis=1)

print("=== 파싱 QA (원본 텍스트 vs 파싱값/근거) - 육안 검증용 ===")
print(
    raw[["gongu", "road_name", "sub_section", "lane_method_text", "lane_remain_ratio", "parse_source"]]
    .to_string(index=False)
)
print("\n파싱 방식별 건수:")
print(raw["parse_source"].value_counts())
print(f"\nfallback(파싱 실패) 비율: {(raw['parse_source']=='fallback_default').mean():.1%}")


=== 파싱 QA (원본 텍스트 vs 파싱값/근거) - 육안 검증용 ===
 gongu       road_name sub_section                lane_method_text  lane_remain_ratio       parse_source
     1            신탄진로         NaN                     편도3차로 중 1차로             0.6667   stage1_full_text
     1             계족로         1구간                            부분통제             0.5000   fallback_default
     1             계족로         2구간                            부분통제             0.5000   fallback_default
     1            신탄진로         3구간                            부분통제             0.5000   fallback_default
     2             계족로       1~4구간                   편도3~5차로 중 1차로             0.6667   stage1_full_text
     2             계족로    법동네거리 일원                       법동방향 2개차로             0.3333 stage2_lane_lookup
     2            한밭대로         NaN    왕복10차로중2 / 왕복8차로중2 / 편도5차로중2             0.6000   stage1_full_text
     3            한밭대로         NaN              왕복 11개차로 중 편도 3개차로             0.7273   stage1_full_text
     4       

In [8]:
# ==================================================================
# PART B-4. 공사개요(총괄) 표 - 8/11공구 fallback + 착공일 교차검증용
# ==================================================================
# 통제현황.xlsx에는 8/11공구가 누락되어 있으므로, 사용자가 제공한 "공사개요(총괄)"
# 표(14개 공구 전체 착공~준공일)를 수동으로 입력해두고, 다음 용도로 쓴다.
#   1) 각 공구의 착공일이 공구_구간_착공일.csv와 일치하는지 교차검증
#   2) 통제현황에 없는 8/11공구에 대해, 착공~준공 전체 기간을 단일 통제 구간으로 간주하고
#      보수적 기본값(0.5)을 적용(network 팀의 파싱불가 fallback 관습과 동일)

gongu_overview = pd.DataFrame(
    [
        (1, "2024-12-06", "2027-03-07"),
        (2, "2024-12-26", "2027-06-25"),
        (3, "2025-09-15", "2028-07-14"),
        (4, "2025-08-25", "2028-05-23"),
        (5, "2025-07-31", "2027-10-30"),
        (6, "2025-05-20", "2028-05-18"),
        (7, "2024-12-27", "2027-08-26"),
        (8, "2025-05-13", "2028-05-11"),
        (9, "2025-07-14", "2028-06-12"),
        (10, "2025-03-07", "2028-09-02"),
        (11, "2025-09-30", "2028-09-28"),
        (12, "2025-09-01", "2028-08-30"),
        (13, "2025-02-28", "2028-06-16"),
        (14, "2025-07-31", "2027-12-27"),
    ],
    columns=["gongu", "start_date", "end_date"],
)
gongu_overview["start_date"] = pd.to_datetime(gongu_overview["start_date"])
gongu_overview["end_date"] = pd.to_datetime(gongu_overview["end_date"])

# 교차검증: 공구_구간_착공일.csv의 공구별 최소 착공일과 이 표의 착공일이 일치하는지 확인
check = gongu_map.groupby("공구")["착공일"].min().reset_index()
check = check.merge(gongu_overview, left_on="공구", right_on="gongu")
check["일치"] = check["착공일"] == check["start_date"]
print("공구_구간_착공일.csv vs 공사개요 총괄 착공일 교차검증:")
print(check[["공구", "착공일", "start_date", "일치"]])
assert check["일치"].all(), "착공일 불일치 발견 - 원인 확인 필요"

MISSING_GONGU = sorted(set(range(1, 15)) - set(raw["gongu"].unique()))
print(f"\n통제현황에 없어 fallback 적용할 공구: {MISSING_GONGU}")

fallback_events = gongu_overview[gongu_overview["gongu"].isin(MISSING_GONGU)].copy()
fallback_events["lane_remain_ratio"] = FALLBACK_RATIO
fallback_events["road_name"] = "(공사개요 fallback)"
fallback_events

공구_구간_착공일.csv vs 공사개요 총괄 착공일 교차검증:
    공구        착공일 start_date    일치
0    1 2024-12-06 2024-12-06  True
1    2 2024-12-26 2024-12-26  True
2    3 2025-09-15 2025-09-15  True
3    4 2025-08-25 2025-08-25  True
4    5 2025-07-31 2025-07-31  True
5    6 2025-05-20 2025-05-20  True
6    7 2024-12-27 2024-12-27  True
7    8 2025-05-13 2025-05-13  True
8    9 2025-07-14 2025-07-14  True
9   10 2025-03-07 2025-03-07  True
10  11 2025-09-30 2025-09-30  True
11  12 2025-09-01 2025-09-01  True
12  13 2025-02-28 2025-02-28  True
13  14 2025-07-31 2025-07-31  True

통제현황에 없어 fallback 적용할 공구: [8, 11]


,gongu,start_date,end_date,lane_remain_ratio,road_name
7,8,2025-05-13,2028-05-11,0.5,(공사개요 fallback)
10,11,2025-09-30,2028-09-28,0.5,(공사개요 fallback)


In [9]:
# ==================================================================
# PART B-5. 공구 이벤트 -> segment_id 레벨 이벤트로 확장
# ==================================================================
# 도로명/위치 텍스트로 정밀 매칭하지 않고, 공구 단위로 통제 이벤트를 broadcast하는
# 단순화된 접근(지난 대화에서 상의된 범위)을 채택한다. 같은 공구 안의 여러 segment_id는
# 모두 그 공구의 모든 통제 이벤트를 공유한다(정밀도는 다소 낮지만 단순하고 보수적).

control_events = raw[["gongu", "road_name", "start_date", "end_date", "lane_remain_ratio"]].dropna(
    subset=["start_date", "end_date"]
)
control_events = pd.concat(
    [control_events, fallback_events[["gongu", "road_name", "start_date", "end_date", "lane_remain_ratio"]]],
    ignore_index=True,
)

# 공구 -> segment_id 다수 매핑으로 확장
segment_events = control_events.merge(gongu_map[["공구", "segment_id"]], left_on="gongu", right_on="공구")
segment_events = segment_events[["segment_id", "road_name", "start_date", "end_date", "lane_remain_ratio"]]

print(f"최종 이벤트 수: {len(segment_events)} (segment_id {segment_events['segment_id'].nunique()}개 커버)")
segment_events.head(10)

최종 이벤트 수: 89 (segment_id 45개 커버)


,segment_id,road_name,start_date,end_date,lane_remain_ratio
0,SEG_43_242_243,신탄진로,2025-03-25,2025-06-30,0.6667
1,SEG_44_243_244,신탄진로,2025-03-25,2025-06-30,0.6667
2,SEG_43_242_243,계족로,2025-05-08,2026-11-30,0.5000
3,SEG_44_243_244,계족로,2025-05-08,2026-11-30,0.5000
4,SEG_43_242_243,계족로,2025-08-25,2026-11-30,0.5000
5,SEG_44_243_244,계족로,2025-08-25,2026-11-30,0.5000
6,SEG_43_242_243,신탄진로,2026-01-15,2026-11-30,0.5000
7,SEG_44_243_244,신탄진로,2026-01-15,2026-11-30,0.5000
8,SEG_41_212_241,계족로,2026-05-08,2026-08-31,0.6667
9,SEG_42_241_242,계족로,2026-05-08,2026-08-31,0.6667


In [10]:
# ==================================================================
# PART B-6. (segment_id, date) 일별 lane_remain_ratio 룩업테이블 생성
# ==================================================================
# 각 이벤트를 해당 기간의 일별 row로 풀어서(explode), 같은 (segment_id, date)에
# 여러 이벤트가 겹치면 최솟값(가장 심각한 통제)을 채택한다. 이벤트가 없는
# 날짜/구간은 1.0(자유흐름, 미착공 또는 통제 공백기)으로 채운다.

speed_df = pl.read_parquet(SPEED_PATH)
date_min = speed_df["timestamp"].min().date()
date_max = speed_df["timestamp"].max().date()
print(f"날짜 범위(속도 데이터 기준): {date_min} ~ {date_max}")

all_dates = pd.date_range(date_min, date_max, freq="1D")
all_segment_ids = sorted(gongu_map["segment_id"].unique())

expanded_rows = []
for row in segment_events.itertuples(index=False):
    ev_start = max(row.start_date, all_dates[0])
    ev_end = min(row.end_date, all_dates[-1])
    if ev_start > ev_end:
        continue
    ev_dates = pd.date_range(ev_start, ev_end, freq="1D")
    expanded_rows.append(
        pd.DataFrame({"segment_id": row.segment_id, "date": ev_dates, "lane_remain_ratio": row.lane_remain_ratio})
    )

expanded = pd.concat(expanded_rows, ignore_index=True)
daily_min = expanded.groupby(["segment_id", "date"], as_index=False)["lane_remain_ratio"].min()

# 전체 (segment_id x date) 그리드에 left join, 비어있는 곳은 1.0(자유흐름)
full_grid = pd.MultiIndex.from_product([all_segment_ids, all_dates], names=["segment_id", "date"]).to_frame(
    index=False
)
daily_lookup = full_grid.merge(daily_min, on=["segment_id", "date"], how="left")
daily_lookup["lane_remain_ratio"] = daily_lookup["lane_remain_ratio"].fillna(1.0)

print(f"일별 룩업테이블 shape: {daily_lookup.shape}")
print(f"통제 중(ratio<1.0) 비율: {(daily_lookup['lane_remain_ratio'] < 1.0).mean():.1%}")

pl.from_pandas(daily_lookup).write_parquet(OUTPUT_DIR / "construction_lane_ratio_daily.parquet")
print(f"저장 완료: {(OUTPUT_DIR / 'construction_lane_ratio_daily.parquet').resolve()}")

날짜 범위(속도 데이터 기준): 2024-10-01 ~ 2026-07-01
일별 룩업테이블 shape: (28755, 3)
통제 중(ratio<1.0) 비율: 32.3%
저장 완료: C:\Users\6152\Desktop\물류\output\features\construction_lane_ratio_daily.parquet


In [11]:
# ==================================================================
# PART B-7. 검증 - 샘플 segment_id의 시간대별 비율 추이 확인
# ==================================================================

sample_segments = ["SEG_43_242_243", "SEG_27_227_228", "SEG_01_201_202"]  # 1공구 / 8공구(fallback) / 12공구

for seg in sample_segments:
    sub = daily_lookup[daily_lookup["segment_id"] == seg].sort_values("date")
    n_days_under_control = (sub["lane_remain_ratio"] < 1.0).sum()
    print(f"{seg}: 통제 일수 {n_days_under_control}/{len(sub)}일, 비율 분포 = {sub['lane_remain_ratio'].value_counts().to_dict()}")

print("\n공구별(segment_id 기준) 평균 통제 일수:")
summary = (
    daily_lookup.assign(under_control=daily_lookup["lane_remain_ratio"] < 1.0)
    .groupby("segment_id")["under_control"]
    .sum()
    .sort_values(ascending=False)
)
print(summary.head(15))

SEG_43_242_243: 통제 일수 464/639일, 비율 분포 = {0.5: 420, 1.0: 175, 0.6667: 44}
SEG_27_227_228: 통제 일수 415/639일, 비율 분포 = {0.5: 415, 1.0: 224}
SEG_01_201_202: 통제 일수 185/639일, 비율 분포 = {1.0: 454, 0.3333: 185}

공구별(segment_id 기준) 평균 통제 일수:
segment_id
SEG_44_243_244    464
SEG_43_242_243    464
SEG_26_226_227    433
SEG_25_225_226    433
SEG_24_224_225    433
SEG_23_223_224    433
SEG_27_227_228    415
SEG_29_229_230    415
SEG_31_231_232    415
SEG_30_230_231    415
SEG_28_228_229    415
SEG_42_241_242    373
SEG_35_235_236    373
SEG_41_212_241    373
SEG_36_236_237    373
Name: under_control, dtype: int64


In [12]:
"""
사용법 요약 (후속 XGBoost 피처 매트릭스 조립 시)

1. output/features/network_features.parquet
   - key: segment_key (=segment_id + "_" + direction)
   - 정적(1행/segment_key) 피처. timestamp 없이 segment_key로만 join.

2. output/features/construction_lane_ratio_daily.parquet
   - key: segment_id (방향 구분 없음), date
   - 피처 매트릭스의 timestamp를 date로 날짜만 추출해 join
     (같은 segment_id의 AB/BA 양쪽 모두 동일한 값을 받음 - 물리적으로 같은 도로이므로 합리적)

주의:
  - lane_remain_ratio 파싱은 자동 정규식 기반이라 100% 정확하지 않을 수 있음.
    PART B-3 QA 출력을 한 번 육안으로 확인하고, 이상한 값은 수동 보정 권장.
  - 8/11공구는 정확한 차로 감소 정보가 없어 착공~준공 전 기간 전체에 보수적 기본값(0.5)을
    일괄 적용함 - 실제로는 그 안에서도 통제 강도가 달라지는 구간이 있을 수 있으므로 참고용.
  - betweenness_pre/during은 단일 스냅샷(ITS 실측 2026-07-07 기준) 기반 정적 피처.
    Train 기간에 따른 재계산이 필요하지 않은 종류의 피처임(우리 속도 시계열과 무관한 외부 소스).
"""
print("완료")

완료
